# LEVIR-CD | Full-data U-Net baseline (M5 / MPS)

**DATA 255 term project  -  reproducible training notebook**

This notebook uses the previously verified split manifest. It trains a **six-channel U-Net** on the complete **445 training image pairs** and validates against the complete **64 validation pairs**, with 256 × 256 nonoverlapping paired tiles. It does **not** read, inspect, or evaluate test images or labels.

**Recommended sequence:**
1. Run all cells with `RUN_MODE = "benchmark"` (one complete epoch, not a tiny subset). Record the epoch time.
2. If speed/memory are acceptable, change `RUN_MODE = "full"` and run all cells again (a new model, up to 30 epochs, early stopping).
3. Share `metrics.csv`, `summary.json`, `learning_curves.png`, `validation_gallery.png` and the executed notebook. Keep checkpoint weights locally or in the team repository's artifact storage.

> The benchmark and full run are intentionally separate training runs. A 1-epoch benchmark is **not** a meaningful model-quality estimate. The full run restarts from seed 1511 for a fair baseline. We will use the same data protocol for a subsequent Siamese U-Net comparison.

## 1. Configuration

Set paths and run mode here. Both `benchmark` and `full` use **all** training/validation tiles. `NUM_WORKERS = 0` is a safe default for local Mac Jupyter sessions; later increase only if you have tested it. Keep the official test split untouched.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import os, json, time, random, math


# Resolve the project root from Jupyter's working directory, unless overridden.
_here = Path.cwd().resolve()
_default_project = _here.parent if _here.name == "notebooks" else _here
DEFAULT_PROJECT_ROOT = Path(os.environ.get("LEVIR_PROJECT_ROOT", str(_default_project))).expanduser().resolve()

@dataclass
class Config:
    PROJECT_ROOT: str = str(DEFAULT_PROJECT_ROOT)
    RUN_MODE: str = "benchmark"   # "benchmark" = 1 full-data epoch; "full" = up to 30 epochs
    SEED: int = 1511
    TILE_SIZE: int = 256
    BATCH_SIZE: int = 4           # Start with 4 for 16GB M5; try 2 if memory pressure
    NUM_WORKERS: int = 0
    BASE_CHANNELS: int = 16       # Lightweight U-Net for MPS; same for all comparisons
    LEARNING_RATE: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    FULL_EPOCHS: int = 30
    EARLY_STOP_PATIENCE: int = 8
    MIN_EPOCHS: int = 12
    MIN_F1_IMPROVEMENT: float = 1e-4
    THRESHOLD: float = 0.5       # Fixed a priori; no test-set tuning

cfg = Config()
cfg.RUN_MODE = "benchmark"  # Change to "full" for the documented 30-epoch run
assert cfg.RUN_MODE in ("benchmark", "full")
assert cfg.TILE_SIZE == 256, "Use 256 initially so train and validation cover all 1024x1024 images."
assert cfg.BATCH_SIZE >= 1
PROJECT_ROOT = Path(cfg.PROJECT_ROOT).expanduser()
MANIFEST = PROJECT_ROOT / "results/levir_audit/split_manifest.csv"
RUN_DIR = PROJECT_ROOT / "results/unet_baseline" / f"{cfg.RUN_MODE}_seed{cfg.SEED}_tile{cfg.TILE_SIZE}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run mode:", cfg.RUN_MODE)
print("Manifest:", MANIFEST)
print("Output directory:", RUN_DIR)
assert MANIFEST.is_file(), "First run the LEVIR-CD audit notebook to generate split_manifest.csv"

## 2. Load and verify the official train / validation manifest

The manifest is produced by the audit notebook you already ran. No file from `test` is loaded in this training notebook. Every original 1024 × 1024 pair contributes sixteen 256 × 256 tiles. The split happens **before** tiling, preventing patches from the same original image from crossing splits.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

manifest = pd.read_csv(MANIFEST)
required_columns = {"split", "stem", "image_A", "image_B", "mask"}
assert required_columns.issubset(manifest.columns), f"Missing columns: {required_columns - set(manifest.columns)}"
assert manifest["split"].value_counts().get("train", 0) == 445, "Expected 445 original training pairs"
assert manifest["split"].value_counts().get("val", 0) == 64, "Expected 64 original validation pairs"

train_df = manifest.loc[manifest.split == "train"].sort_values("stem").reset_index(drop=True)
val_df = manifest.loc[manifest.split == "val"].sort_values("stem").reset_index(drop=True)
assert set(train_df.stem).isdisjoint(set(val_df.stem))
for frame in (train_df, val_df):
    assert frame.stem.is_unique
    for key in ("image_A", "image_B", "mask"):
        missing = [p for p in frame[key] if not Path(p).is_file()]
        if missing:
            raise FileNotFoundError(f"Missing {key} files: {missing[:3]}")
print("Original pairs: train=", len(train_df), ", val=", len(val_df))
print("Expected tiles: train=", len(train_df)*16, ", val=", len(val_df)*16)
print("No test records are used below.")

## 3. Device, seed, reproducibility

On your Mac M5 this should print `mps`. For a controlled comparison, keep seed, architecture width, preprocessing, loss, and learning rate unchanged. Some MPS kernels can remain nondeterministic across machines or versions; record the actual software and hardware environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device, "| processor:", os.uname().machine)
print("MPS enabled:", torch.backends.mps.is_available())
# CUDA-specific mixed precision is intentionally not used on MPS.

def sync_device():
    if device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

## 4. Paired 256 × 256 tile dataset

Each sample crops the same tile coordinates from **before, after and mask**; geometric augmentation is synchronized across all three. Labels are binarized at `>=128` to match your audit. The loader does not save millions of extra image files. It opens a source image when needed, so a complete benchmark is the best way to judge whether training locally is fast enough.

In [ ]:
class ChangeTiles(Dataset):
    def __init__(self, frame, size=256, augment=False):
        self.frame = frame.reset_index(drop=True).copy()
        self.size = int(size)
        self.augment = bool(augment)
        self.tiles = []
        for row_i, row in self.frame.iterrows():
            # Source audit established 1024x1024; spot-check every shape during retrieval.
            for y in range(0, 1024, self.size):
                for x in range(0, 1024, self.size):
                    self.tiles.append((row_i, x, y))

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        row_i, x, y = self.tiles[idx]
        row = self.frame.iloc[row_i]
        rect = (x, y, x + self.size, y + self.size)
        with Image.open(row.image_A) as source:
            if source.size != (1024, 1024): raise ValueError(f"Bad A size: {row.image_A}")
            a = np.array(source.convert("RGB").crop(rect), dtype=np.uint8)
        with Image.open(row.image_B) as source:
            if source.size != (1024, 1024): raise ValueError(f"Bad B size: {row.image_B}")
            b = np.array(source.convert("RGB").crop(rect), dtype=np.uint8)
        with Image.open(row["mask"]) as source:
            if source.size != (1024, 1024): raise ValueError(f"Bad mask size: {row['mask']}")
            mask = np.asarray(source.convert("L").crop(rect)) >= 128

        # Identical spatial transforms for before, after and target.
        if self.augment:
            if random.random() < 0.5:
                a, b, mask = a[:, ::-1], b[:, ::-1], mask[:, ::-1]
            if random.random() < 0.5:
                a, b, mask = a[::-1], b[::-1], mask[::-1]
            k = random.randrange(4)
            if k:
                a, b, mask = np.rot90(a, k), np.rot90(b, k), np.rot90(mask, k)

        x_arr = np.concatenate([a, b], axis=2).transpose(2, 0, 1).copy()
        y_arr = mask.astype(np.float32, copy=False)[None].copy()
        x_tensor = torch.from_numpy(x_arr).float().div_(255.0)
        y_tensor = torch.from_numpy(y_arr)
        return x_tensor, y_tensor

train_ds = ChangeTiles(train_df, cfg.TILE_SIZE, augment=True)
val_ds = ChangeTiles(val_df, cfg.TILE_SIZE, augment=False)
assert len(train_ds) == 7120 and len(val_ds) == 1024
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=(device.type=="cuda"))
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=(device.type=="cuda"))
example_x, example_y = val_ds[0]
assert tuple(example_x.shape) == (6, 256, 256)
assert tuple(example_y.shape) == (1, 256, 256)
assert set(torch.unique(example_y).tolist()).issubset({0., 1.})
print("Full-data tile counts:", len(train_ds), "train /", len(val_ds), "val")
print("Batch tensor shapes:", tuple(example_x.shape), tuple(example_y.shape))

## 5. Lightweight U-Net, 6 input channels → 1 change logit

Two RGB images become six input channels. Output is a **single logit map**, not a probability; `sigmoid` is applied for F1/IoU. Standard U-Net encoder/decoder uses skip connections. The base width of 16 is deliberately small to fit in 16 GB unified memory.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=6, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base)
        self.enc2 = ConvBlock(base, base*2)
        self.enc3 = ConvBlock(base*2, base*4)
        self.bottleneck = ConvBlock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, kernel_size=1)
    def forward(self, x):
        a = self.enc1(x)
        b = self.enc2(self.pool(a))
        c = self.enc3(self.pool(b))
        z = self.bottleneck(self.pool(c))
        z = self.dec3(torch.cat([self.up3(z), c], dim=1))
        z = self.dec2(torch.cat([self.up2(z), b], dim=1))
        z = self.dec1(torch.cat([self.up1(z), a], dim=1))
        return self.out(z)

model = UNet(base=cfg.BASE_CHANNELS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE,
                             weight_decay=cfg.WEIGHT_DECAY)
num_params = sum(p.numel() for p in model.parameters())
print(f"U-Net trainable parameters: {num_params:,}")
with torch.no_grad():
    pred = model(torch.zeros(1, 6, 256, 256, device=device))
assert tuple(pred.shape) == (1, 1, 256, 256)
print("Forward-pass OK:", tuple(pred.shape))

## 6. Loss and pooled validation metrics

**Loss:** BCEWithLogits + soft Dice (equal weights). **Primary metric:** changed-class F1/Dice at a fixed 0.5 probability threshold, pooled over **all validation pixels**, not averaged per batch. Secondary metrics: IoU, precision, recall and validation loss. All-zero predictions are recorded as F1=0 when the validation split contains changed pixels.

In [ ]:
def bce_dice_loss(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    dims = (1, 2, 3)
    numerator = 2.0 * (prob * target).sum(dims) + 1.0
    denominator = prob.sum(dims) + target.sum(dims) + 1.0
    dice_loss = 1.0 - (numerator / denominator).mean()
    return bce + dice_loss

@torch.no_grad()
def validate(net, loader, threshold=0.5):
    net.eval()
    tp = fp = fn = 0
    total_loss = total_samples = 0.0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = net(x)
        loss = bce_dice_loss(logits, y)
        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)
        pred = (torch.sigmoid(logits) >= threshold)
        truth = y >= 0.5
        tp += int((pred & truth).sum().item())
        fp += int((pred & ~truth).sum().item())
        fn += int((~pred & truth).sum().item())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * tp / (2*tp + fp + fn) if (2*tp + fp + fn) else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    return dict(val_loss=total_loss / max(total_samples, 1), val_precision=precision,
                val_recall=recall, val_f1=f1, val_iou=iou, tp=tp, fp=fp, fn=fn)

def fit_one_epoch(net, loader):
    net.train()
    total_loss = 0.0
    total_n = 0
    for batch_id, (x, y) in enumerate(loader, 1):
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        output = net(x)
        loss = bce_dice_loss(output, y)
        if not torch.isfinite(loss).item():
            raise RuntimeError(f"Nonfinite loss at batch {batch_id}")
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)
        if batch_id == 1 or batch_id % 250 == 0 or batch_id == len(loader):
            print(f"  train batch {batch_id}/{len(loader)} | loss {loss.item():.4f}", flush=True)
    return total_loss / max(total_n, 1)

## 7. Train, validate, save metrics and checkpoints

`benchmark`: 1 complete training epoch **and** complete validation epoch. `full`: up to 30 complete epochs; best checkpoint selected by validation F1, patience 8 after epoch 12. Both modes save `metrics.csv`, `summary.json`, `best_unet.pt`, and `last_unet.pt` to **different directories**. The benchmark is not a final result, and full mode trains a fresh model from the same seed. Training speed depends on your actual M5 machine and disk performance.

In [ ]:
epochs = 1 if cfg.RUN_MODE == "benchmark" else cfg.FULL_EPOCHS
best_f1 = -1.0
best_epoch = None
bad_epochs = 0
rows = []
started = time.perf_counter()

(RUN_DIR / "config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")
for epoch in range(1, epochs + 1):
    sync_device()
    tick = time.perf_counter()
    print(f"\nEpoch {epoch}/{epochs}  -  full train + full validation", flush=True)
    train_loss = fit_one_epoch(model, train_loader)
    metrics = validate(model, val_loader, cfg.THRESHOLD)
    sync_device()
    elapsed = time.perf_counter() - tick
    row = {"epoch": epoch, "train_loss": train_loss, **metrics, "elapsed_seconds": elapsed}
    rows.append(row)
    pd.DataFrame(rows).to_csv(RUN_DIR / "metrics.csv", index=False)
    print(f"  loss {train_loss:.4f} / val {metrics['val_loss']:.4f} | "
          f"F1 {metrics['val_f1']:.4f} | IoU {metrics['val_iou']:.4f} | "
          f"precision {metrics['val_precision']:.4f} | recall {metrics['val_recall']:.4f} "
          f"| {elapsed/60:.1f} min", flush=True)

    state = {"epoch": epoch, "model_state": model.state_dict(),
             "optimizer_state": optimizer.state_dict(), "config": asdict(cfg),
             "metrics": row, "torch_version": str(torch.__version__)}
    torch.save(state, RUN_DIR / "last_unet.pt")
    if metrics["val_f1"] > best_f1 + cfg.MIN_F1_IMPROVEMENT:
        best_f1 = metrics["val_f1"]
        best_epoch = epoch
        bad_epochs = 0
        torch.save(state, RUN_DIR / "best_unet.pt")
        print("  saved new best checkpoint", flush=True)
    else:
        bad_epochs += 1
    if cfg.RUN_MODE == "full" and epoch >= cfg.MIN_EPOCHS and bad_epochs >= cfg.EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch}; best epoch {best_epoch}")
        break

sync_device()
total_seconds = time.perf_counter() - started
summary = {"run_mode": cfg.RUN_MODE, "completed_epochs": len(rows),
           "best_epoch": best_epoch, "best_val_f1": best_f1,
           "total_minutes": round(total_seconds/60, 2),
           "mean_epoch_minutes": round(sum(r["elapsed_seconds"] for r in rows)/60/len(rows), 2),
           "train_original_pairs": len(train_df), "val_original_pairs": len(val_df),
           "train_tiles": len(train_ds), "val_tiles": len(val_ds),
           "parameters": num_params,
           "threshold": cfg.THRESHOLD, "selected_using": "validation F1 only",
           "test_set_touched": False,
           "note": "One-epoch benchmark is not a final model-quality result" if cfg.RUN_MODE == "benchmark" else "Full U-Net baseline"}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Saved to:", RUN_DIR)

## 8. Visualize learning curves

For a benchmark, the plots contain just one point. For a full run, compare train/validation losses and F1/IoU across epochs. Save the plots for your report.

In [ ]:
history = pd.read_csv(RUN_DIR / "metrics.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.epoch, history.train_loss, "o-", label="Train loss")
axes[0].plot(history.epoch, history.val_loss, "o-", label="Validation loss")
axes[0].set(title="BCE + Dice loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].plot(history.epoch, history.val_f1, "o-", label="Validation F1")
axes[1].plot(history.epoch, history.val_iou, "o-", label="Validation IoU")
axes[1].set(title="Change-pixel segmentation", xlabel="Epoch", ylabel="Score", ylim=(0, 1))
axes[1].legend(); axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(RUN_DIR / "learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Qualitative validation predictions (no test images)

The figure shows the *before*, *after*, ground-truth mask, predicted probability heatmap and thresholded predicted mask for three preselected validation patches (including a no-change patch if available). To avoid cherry-picking, patch selection is based on ground truth only, never on the model's predictions. Predictions use the **best validation checkpoint**.

In [ ]:
# Select two tiles containing changed buildings and one negative tile using labels only.
pos_indices, zero_indices = [], []
for idx, (row_i, x0, y0) in enumerate(val_ds.tiles):
    row = val_ds.frame.iloc[row_i]
    with Image.open(row["mask"]) as im:
        patch = np.asarray(im.convert("L").crop((x0, y0, x0+256, y0+256))) >= 128
    fraction = float(patch.mean())
    if fraction >= 0.03 and len(pos_indices) < 2:
        pos_indices.append(idx)
    elif fraction == 0 and len(zero_indices) < 1:
        zero_indices.append(idx)
    if len(pos_indices) >= 2 and len(zero_indices) >= 1:
        break
chosen = pos_indices + zero_indices
if len(chosen) < 3:
    chosen.extend(i for i in range(len(val_ds)) if i not in chosen)
chosen = chosen[:3]
checkpoint = torch.load(RUN_DIR / "best_unet.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
fig, axes = plt.subplots(len(chosen), 5, figsize=(15, 3.1*len(chosen)), squeeze=False)
for r, i in enumerate(chosen):
    x_tensor, y_tensor = val_ds[i]
    with torch.no_grad():
        probs = torch.sigmoid(model(x_tensor[None].to(device)))[0,0].cpu().numpy()
    xnp = x_tensor.numpy()
    images = [np.moveaxis(xnp[:3],0,-1), np.moveaxis(xnp[3:],0,-1),
              y_tensor[0].numpy(), probs, (probs >= cfg.THRESHOLD).astype(np.float32)]
    titles = ["Before", "After", "Ground truth", "Change probability", "Prediction @ 0.5"]
    for j, (im, title) in enumerate(zip(images,titles)):
        ax = axes[r,j]
        if j in (0, 1): ax.imshow(im)
        else: ax.imshow(im, cmap="magma" if j==3 else "gray", vmin=0, vmax=1)
        ax.set_title(title if r==0 else "", fontsize=10)
        ax.axis("off")
fig.suptitle(f"Validation only  -  U-Net best epoch {checkpoint['epoch']} ({cfg.RUN_MODE})", fontsize=12)
fig.tight_layout()
fig.savefig(RUN_DIR / "validation_gallery.png", dpi=160, bbox_inches="tight")
plt.show()
print("Output files:")
for p in sorted(RUN_DIR.iterdir()):
    print(" ", p.name, f"({p.stat().st_size/(1024**2):.1f} MiB)")

## 10. Interpreting the result and next experiment

**If you ran `benchmark`:** report the epoch time and verify there are no MPS out-of-memory errors. A slow first epoch may include caching/initialization overhead. If feasible, set `RUN_MODE="full"` in **Cell 1** and run all cells again. This starts a new model, from scratch. If it is too slow, use SJSU GPU lab for the full run.

**If you ran `full`:** share `metrics.csv`, `summary.json`, `learning_curves.png`, and `validation_gallery.png`. Save the best checkpoint safely. Do not open or evaluate the test split yet.

**Research next:** train Siamese U-Net with the same splits, patch size, augmentation, budget, seed, and fixed threshold. Then compare pooled validation F1, IoU, recall and inference speed. After choosing your final protocol using validation only, run one held-out test evaluation for both frozen configurations.

**Notes:** Pixel-level train/validation class imbalance was established in your audit. Image tiles from the same source pair remain within a single split. This notebook does not by itself establish geographic independence between split locations; report that limitation or use the official geospatial metadata for an additional audit. Handle academic-only LEVIR-CD licensing and proper paper citation in the final report.